# Train SciBERT contrastive variants (Colab T4)
Runs all four FT variants. Skips b_only / ab_* if `data/generated_queries.jsonl` is empty.
Total ~30-60 min on T4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
rm -rf /content/repo
git clone https://github.com/wswaileh/palestinian-drugs-data.git /content/repo
cd /content/repo && git checkout main
pip install -q -r requirements.txt
# Trainer auto-detects wandb / codecarbon and crashes without keys; remove them.
pip uninstall -q -y wandb codecarbon mlflow comet-ml || true

In [ ]:
%cd /content/repo
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
%%bash
set -e
export WANDB_DISABLED=true
export WANDB_MODE=disabled
for variant in a_only b_only ab_random ab_atc; do
  echo "==== training variant: $variant ===="
  python -m src.training.train --variant $variant \
    --output-dir /content/drive/MyDrive/scibert_ft || echo "variant $variant failed (likely missing queries.jsonl); continuing"
done

In [ ]:
%%bash
ls -la /content/drive/MyDrive/scibert_ft/